# CTAI - YOLO11-seg 直肠肿瘤分割训练

在 Kaggle T4 GPU 上训练 YOLO11n-seg 模型。

**使用方法：**
1. 上传 `直肠癌数据_tiny` 或 `直肠癌数据` 为 Kaggle Dataset
2. 开启 GPU T4 加速器
3. 全部运行即可

In [ ]:
!pip install ultralytics==8.3.2 SimpleITK -q

In [ ]:
import os, glob, random, shutil
import numpy as np
import cv2
import SimpleITK as sitk
from pathlib import Path

# ========== 配置 ==========
# Kaggle dataset 挂载路径，按实际情况改
DATA_DIR = '/kaggle/input/rectal-tumor-ct'  
OUTPUT_DIR = '/kaggle/working/datasets/rectal_tumor_seg'
VAL_RATIO = 0.15
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

## 1. 数据格式转换 (DCM -> YOLO seg)

In [ ]:
def dcm_to_png(dcm_path, wc=40, ww=400):
    img = sitk.ReadImage(dcm_path)
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    if arr.ndim == 3: arr = arr[0]
    lo, hi = wc - ww/2, wc + ww/2
    arr = np.clip(arr, lo, hi)
    return ((arr - lo) / (hi - lo) * 255).astype(np.uint8)

def mask_to_yolo(mask_bin, h, w):
    contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 10: continue
        cnt = cnt.squeeze()
        if cnt.ndim != 2 or len(cnt) < 3: continue
        pts = []
        for x, y in cnt:
            pts.extend([f'{x/w:.6f}', f'{y/h:.6f}'])
        lines.append('0 ' + ' '.join(pts))
    return lines

# 扫描数据目录
patients = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
random.shuffle(patients)
val_n = max(1, int(len(patients) * VAL_RATIO))
val_set = set(patients[:val_n])

for sp in ['train', 'val']:
    os.makedirs(f'{OUTPUT_DIR}/images/{sp}', exist_ok=True)
    os.makedirs(f'{OUTPUT_DIR}/labels/{sp}', exist_ok=True)

stats = {'train': 0, 'val': 0}
for pid in patients:
    split = 'val' if pid in val_set else 'train'
    pdir = os.path.join(DATA_DIR, pid)
    scans = [d for d in os.listdir(pdir) if os.path.isdir(os.path.join(pdir, d)) and '_mask' not in d]
    for sn in scans:
        sdir = os.path.join(pdir, sn)
        mdir = os.path.join(pdir, sn + '_mask')
        if not os.path.isdir(mdir): continue
        for dcm in sorted(glob.glob(os.path.join(sdir, '*.dcm'))):
            fn = os.path.splitext(os.path.basename(dcm))[0]
            mpath = os.path.join(mdir, os.path.basename(dcm))
            if not os.path.isfile(mpath): continue
            img = dcm_to_png(dcm)
            h, w = img.shape[:2]
            img3 = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
            m = sitk.GetArrayFromImage(sitk.ReadImage(mpath))
            if m.ndim == 3: m = m[0]
            mb = (m > 0).astype(np.uint8) * 255
            labels = mask_to_yolo(mb, h, w)
            if not labels: continue  # 跳过无标注切片
            name = f'{pid}_{sn}_{fn}'
            cv2.imwrite(f'{OUTPUT_DIR}/images/{split}/{name}.png', img3)
            with open(f'{OUTPUT_DIR}/labels/{split}/{name}.txt', 'w') as f:
                f.write('\n'.join(labels))
            stats[split] += 1

print(f'转换完成: train={stats["train"]}, val={stats["val"]}')

In [ ]:
# 生成 data.yaml
yaml_content = f"""path: {OUTPUT_DIR}
train: images/train
val: images/val

names:
  0: tumor
"""
with open(f'{OUTPUT_DIR}/data.yaml', 'w') as f:
    f.write(yaml_content)
print('data.yaml 已生成')

## 2. 训练 YOLO11n-seg

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-seg.pt')  # COCO 预训练权重，自动下载

results = model.train(
    data=f'{OUTPUT_DIR}/data.yaml',
    epochs=200,
    imgsz=512,
    batch=8,
    patience=30,
    device=0,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    close_mosaic=10,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.2,
    degrees=15.0, translate=0.1, scale=0.3,
    flipud=0.5, fliplr=0.5,
    mosaic=0.5, mixup=0.0,
    single_cls=True,
    project='/kaggle/working/runs',
    name='rectal_tumor',
    save=True,
)

print(f'训练完成！最佳模型: {results.save_dir}/weights/best.pt')

## 3. 验证 & 指标输出

In [ ]:
# 加载最佳权重验证
best_model = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = best_model.val(data=f'{OUTPUT_DIR}/data.yaml', imgsz=512, device=0)

print('='*50)
print('验证集指标:')
print(f'  mAP@50 (mask):    {metrics.seg.map50:.3f}')
print(f'  mAP@50-95 (mask): {metrics.seg.map:.3f}')
print(f'  Precision:        {metrics.seg.mp:.3f}')
print(f'  Recall:           {metrics.seg.mr:.3f}')
print('='*50)

In [ ]:
# 计算 Dice 系数
import torch

def calc_dice(pred_mask, gt_mask):
    intersection = (pred_mask & gt_mask).sum()
    return (2.0 * intersection) / (pred_mask.sum() + gt_mask.sum() + 1e-8)

val_imgs = sorted(glob.glob(f'{OUTPUT_DIR}/images/val/*.png'))
dice_scores = []

for img_path in val_imgs:
    name = Path(img_path).stem
    label_path = f'{OUTPUT_DIR}/labels/val/{name}.txt'
    
    # 推理
    res = best_model.predict(img_path, imgsz=512, conf=0.25, retina_masks=True, verbose=False)
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    
    # 预测 mask
    pred = np.zeros((h, w), dtype=np.uint8)
    if res[0].masks is not None:
        for m in res[0].masks.data.cpu().numpy():
            if m.shape[:2] != (h, w):
                m = cv2.resize(m, (w, h))
            pred = np.maximum(pred, (m > 0.5).astype(np.uint8))
    
    # GT mask (从 YOLO 标签恢复)
    gt = np.zeros((h, w), dtype=np.uint8)
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()[1:]  # 去掉 class id
                pts = np.array([float(x) for x in parts]).reshape(-1, 2)
                pts[:, 0] *= w
                pts[:, 1] *= h
                cv2.fillPoly(gt, [pts.astype(np.int32)], 1)
    
    dice_scores.append(calc_dice(pred.astype(bool), gt.astype(bool)))

print(f'Dice 系数 (验证集均值): {np.mean(dice_scores):.3f} ± {np.std(dice_scores):.3f}')

## 4. 推理可视化 (论文图 5-3 素材)

In [ ]:
import matplotlib.pyplot as plt

# 挑 4 个 case 可视化
samples = val_imgs[:4]
fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4*len(samples)))
titles = ['原始CT', 'Ground Truth', 'YOLO11预测']

for i, img_path in enumerate(samples):
    name = Path(img_path).stem
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    
    # GT
    gt = np.zeros((h, w), dtype=np.uint8)
    lp = f'{OUTPUT_DIR}/labels/val/{name}.txt'
    if os.path.exists(lp):
        with open(lp) as f:
            for line in f:
                parts = line.strip().split()[1:]
                pts = np.array([float(x) for x in parts]).reshape(-1, 2)
                pts[:, 0] *= w; pts[:, 1] *= h
                cv2.fillPoly(gt, [pts.astype(np.int32)], 255)
    
    # Pred
    res = best_model.predict(img_path, imgsz=512, conf=0.25, retina_masks=True, verbose=False)
    pred = np.zeros((h, w), dtype=np.uint8)
    if res[0].masks is not None:
        for m in res[0].masks.data.cpu().numpy():
            if m.shape[:2] != (h, w): m = cv2.resize(m, (w, h))
            pred = np.maximum(pred, ((m > 0.5) * 255).astype(np.uint8))
    
    for j, data in enumerate([img, gt, pred]):
        axes[i][j].imshow(data, cmap='gray')
        axes[i][j].set_title(titles[j])
        axes[i][j].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/fig5_3_validation_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 5-3 素材已保存')

## 5. 打包下载

训练产物在 `/kaggle/working/runs/rectal_tumor/` 下：
- `weights/best.pt` — 最佳模型权重
- `results.csv` — 逐 epoch 指标
- `results.png` — loss + metrics 曲线（论文图 5-1, 5-2 素材）
- `val_batch0_pred.jpg` — 验证集预测样例

In [ ]:
# 打包所有产物
shutil.make_archive('/kaggle/working/ctai_yolo_results', 'zip', '/kaggle/working/runs/rectal_tumor')
print('已打包: /kaggle/working/ctai_yolo_results.zip')